# Accumulation modes: atomic, deterministic, simdgroup

`MetalTreeExplainer` can accumulate per-element contributions three ways:

| mode | what it does | when to use |
|---|---|---|
| `atomic` *(default)* | float atomics scatter contributions directly | throughput |
| `deterministic` | fixed-shape two-stage Kahan reduction over canonical slots | bit-repeatable pipelines, tighter reduction accuracy |
| `simdgroup` | SIMD-group pre-aggregation before the atomic | experimentation on other GPUs/models |

Atomic mode is fastest but floating-point addition order varies run to run, so
outputs jitter at the ~1e-6 level. Deterministic mode makes the summation shape a
pure function of the model — **identical bits every run, at every tile size**.

In [1]:
import hashlib
import time
import warnings

import numpy as np

warnings.filterwarnings("ignore", message="A NumPy version")
import xgboost as xgb  # noqa: E402  (after the filter, on purpose)

from metal_treeshap import MetalTreeExplainer  # noqa: E402

rng = np.random.default_rng(11)
X_fit = rng.standard_normal((30_000, 10)).astype(np.float32)
X_fit[rng.random(X_fit.shape) < 0.1] = np.nan
y = np.nansum(X_fit[:, :5], axis=1) + 0.1 * rng.standard_normal(len(X_fit))
booster = xgb.train(
    {"objective": "reg:squarederror", "max_depth": 7, "eta": 0.1,
     "tree_method": "hist", "seed": 11},
    xgb.DMatrix(X_fit, label=y), num_boost_round=300,
)
X = rng.standard_normal((8_000, 10)).astype(np.float32)
X[rng.random(X.shape) < 0.1] = np.nan

atomic = MetalTreeExplainer.from_xgboost(booster)  # accumulation="atomic"
deterministic = MetalTreeExplainer.from_xgboost(booster, accumulation="deterministic")
print("compiled one explainer per mode (300 trees, depth 7; explaining 8,000 rows)")

compiled one explainer per mode (300 trees, depth 7; explaining 8,000 rows)


## Run-to-run repeatability

In [2]:
def digest(phis):
    return hashlib.sha256(phis.tobytes()).hexdigest()[:16]

for name, explainer in [("atomic", atomic), ("deterministic", deterministic)]:
    a, b = explainer.shap_values(X), explainer.shap_values(X)
    print(f"{name:>13}: bitwise equal = {np.array_equal(a, b)!s:5}  "
          f"max |delta| = {float(np.abs(a - b).max()):.2e}  "
          f"hashes = {digest(a)} / {digest(b)}")

       atomic: bitwise equal = False  max |delta| = 1.50e-05  hashes = 0d3bf3ff5a134eda / 1f5f4571dbac185a


deterministic: bitwise equal = True   max |delta| = 0.00e+00  hashes = 1e69819988d40360 / 1e69819988d40360


Atomic outputs differ in the last float bits between runs; deterministic outputs are
**hash-identical**. The jitter is far below any statistical meaning in the
attributions — determinism matters when you diff artifacts, cache results, or audit
pipelines.

## What does determinism cost?

The deterministic reducer is a fixed-shape two-stage Kahan sum (256-slot chunks),
which also improves reduction accuracy. A quick paired timing on this workload:

In [3]:
samples = {"atomic": [], "deterministic": []}
for _ in range(5):
    for name, explainer in [("atomic", atomic), ("deterministic", deterministic)]:
        t0 = time.perf_counter()
        explainer.shap_values(X)
        samples[name].append(time.perf_counter() - t0)
a_med = float(np.median(samples["atomic"]))
d_med = float(np.median(samples["deterministic"]))
print(f"atomic        : {a_med * 1e3:7.1f} ms")
print(f"deterministic : {d_med * 1e3:7.1f} ms  ({d_med / a_med:.2f}x atomic)")

atomic        :   331.4 ms
deterministic :   361.4 ms  (1.09x atomic)


## Introspection and tuning

`last_timings` reports what the most recent call actually did: GPU time, whether the
input and output took the zero-copy paths, and — in deterministic mode — the scratch
footprint and row tiling. The scratch budget is a hard cap: shrinking it tiles the
batch into more, smaller dispatches **without changing a single output bit**.

In [4]:
deterministic.shap_values(X)
{k: v for k, v in deterministic.last_timings.items() if v}

{'upload_s': 1.5125e-05,
 'encode_s': 6.9917e-05,
 'gpu_s': 0.2970308749936521,
 'total_s': 0.297750333,
 'output_zero_copy': True,
 'dispatched': True,
 'deterministic_scratch_bytes': 267920352,
 'deterministic_scratch_capacity_bytes': 267920352,
 'deterministic_tile_rows': 434,
 'deterministic_tiles': 19}

In [5]:
budget_8 = MetalTreeExplainer.from_xgboost(
    booster, accumulation="deterministic", deterministic_scratch_mib=8)
reference = deterministic.shap_values(X)
tiled = budget_8.shap_values(X)
t = budget_8.last_timings
print(f"8 MiB budget -> {t['deterministic_tiles']} tiles of "
      f"{t['deterministic_tile_rows']} rows "
      f"(default budget used {deterministic.last_timings['deterministic_tiles']} tile)")
print(f"bit-identical to the default-budget result: {np.array_equal(reference, tiled)}")

8 MiB budget -> 616 tiles of 13 rows (default budget used 19 tile)
bit-identical to the default-budget result: True


Other knobs, all keyword arguments of `from_xgboost` / `from_paths`:

- `rows_per_simdgroup`, `threads_per_threadgroup` — dispatch shape (defaults 256/256
  were tuned on M4 Max);
- `atomic_tile_rows` — split atomic dispatches into row tiles (0 = full batch);
- `model_storage="private"` — blit model buffers to GPU-private memory;
- `explainer.trim_buffers()` — release persistent buffers after an unusually large
  batch (they regrow on demand).

In [6]:
atomic.trim_buffers()  # returns the peak-batch allocation; buffers regrow on demand
print("trimmed persistent buffers")

trimmed persistent buffers
